# Transfer Learning with ResNet-50 on CIFAR-10 / CIFAR-100 using PyTorch

**Google Colab Ready Notebook**

This notebook implements **ResNet-50 pretrained on ImageNet** for **CIFAR-10 or CIFAR-100** classification using **transfer learning** in **PyTorch**. It includes:

- Data loading, validation splitting, and augmentation
- ResNet-50 model modification
- Three distinct freezing strategies (Fully Frozen, Partially Frozen, Fully Trainable)
- Early stopping and validation check points
- Learning rate hyperparameter tuning
- Learning curves, confusion matrices, and classification reports
- Model saving/loading and inference on sample test images

> **Note on Fast-Run Mode:** To allow quick verification of this notebook on CPU (such as local environments), we have added a `FAST_RUN = True` setting. It will use a small subset of the dataset and train for single epochs to test the entire pipeline in seconds. Turn `FAST_RUN = False` and select GPU on Colab for the full training run!

## 1. Install Required Libraries

In [ ]:
!pip -q install seaborn scikit-learn tqdm pandas matplotlib

## 2. Imports and Configuration

In [ ]:
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

try:
    get_ipython()
    from tqdm.notebook import tqdm
except (NameError, ImportError):
    from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights

In [ ]:
# =========================
# Configuration
# =========================

DATASET_NAME = "CIFAR10"   # Change to "CIFAR100" if required

BATCH_SIZE = 64
IMAGE_SIZE = 224
NUM_WORKERS = 0 if os.name == 'nt' else 2
VAL_SPLIT = 0.1
RANDOM_SEED = 42

MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
WEIGHT_DECAY = 1e-4

MODEL_SAVE_DIR = "./saved_models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ImageNet normalization values for pretrained ResNet-50
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Hyperparameter candidates for tuning
LR_CANDIDATES = {
    "fully_frozen": [1e-3, 5e-4],
    "partially_frozen": [3e-4, 1e-4],
    "fully_trainable": [1e-4, 3e-5]
}
TUNE_EPOCHS = 3  

# =========================
# FAST_RUN Toggle
# =========================
# If True, uses a tiny subset of the dataset and 1 epoch for super-quick validation.
# Defaults to True on CPU (for local machine checks) and False if CUDA is available.
FAST_RUN = not torch.cuda.is_available()
print(f"FAST_RUN mode: {FAST_RUN} (CPU fallback/Dry run)")

In [ ]:
# =========================
# Reproducibility
# =========================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

## 3. Data Loading and Preprocessing

- Resize CIFAR images to **224 × 224**
- Normalize using **ImageNet mean and standard deviation**
- Create **train**, **validation**, and **test** loaders

In [ ]:
# =========================
# Data transforms
# =========================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [ ]:
# =========================
# Dataset loader function
# =========================

def get_cifar_datasets(dataset_name="CIFAR10", fast_run=False):
    if dataset_name == "CIFAR10":
        train_dataset_aug = datasets.CIFAR10(
            root="./data", train=True, transform=train_transform, download=True
        )
        train_dataset_eval = datasets.CIFAR10(
            root="./data", train=True, transform=test_transform, download=True
        )
        test_dataset = datasets.CIFAR10(
            root="./data", train=False, transform=test_transform, download=True
        )
        num_classes = 10

    elif dataset_name == "CIFAR100":
        train_dataset_aug = datasets.CIFAR100(
            root="./data", train=True, transform=train_transform, download=True
        )
        train_dataset_eval = datasets.CIFAR100(
            root="./data", train=True, transform=test_transform, download=True
        )
        test_dataset = datasets.CIFAR100(
            root="./data", train=False, transform=test_transform, download=True
        )
        num_classes = 100

    else:
        raise ValueError("dataset_name must be either 'CIFAR10' or 'CIFAR100'")

    class_names = train_dataset_aug.classes

    num_train = len(train_dataset_aug)
    indices = np.arange(num_train)
    np.random.shuffle(indices)

    split = int(np.floor(VAL_SPLIT * num_train))
    val_idx = indices[:split]
    train_idx = indices[split:]

    if fast_run:
        train_idx = train_idx[:200]
        val_idx = val_idx[:50]
        test_dataset = Subset(test_dataset, range(50))

    train_subset = Subset(train_dataset_aug, train_idx)
    val_subset = Subset(train_dataset_eval, val_idx)

    return train_subset, val_subset, test_dataset, class_names, num_classes

In [ ]:
train_dataset, val_dataset, test_dataset, class_names, NUM_CLASSES = get_cifar_datasets(DATASET_NAME, fast_run=FAST_RUN)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False
)

print(f"Dataset: {DATASET_NAME}")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {NUM_CLASSES}")

## 4. Visualize Sample Images

In [ ]:
def imshow_tensor(tensor_img, title=None):
    # De-normalize and show PyTorch tensor
    img = tensor_img.numpy().transpose((1, 2, 0))
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    if title is not None:
        plt.title(title)
    plt.axis('off')

# Fetch a batch and visualize a few samples
dataiter = iter(train_loader)
images, labels = next(dataiter)

plt.figure(figsize=(12, 6))
for i in range(min(6, len(images))):
    plt.subplot(2, 3, i + 1)
    imshow_tensor(images[i], title=class_names[labels[i]])
plt.tight_layout()
plt.show()

## 5. Model Architecture

We load pretrained **ResNet-50**, replace the final fully connected layer, and control which layers are frozen or trainable.

### Freezing Strategy

#### Fully Frozen
- **Frozen:** `conv1`, `bn1`, `layer1`, `layer2`, `layer3`, `layer4`
- **Trainable:** `fc`

#### Partially Frozen
- **Frozen:** `conv1`, `bn1`, `layer1`, `layer2`
- **Trainable:** `layer3`, `layer4`, `fc`

#### Fully Trainable
- **Frozen:** None
- **Trainable:** All layers

In [ ]:
# =========================
# Model builder
# =========================

def build_resnet50_transfer(num_classes, mode="partially_frozen"):
    """
    mode options:
    - 'fully_frozen'
    - 'partially_frozen'
    - 'fully_trainable'
    """
    weights = ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)

    # Replace final fully connected layer
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    if mode == "fully_frozen":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True

    elif mode == "partially_frozen":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.layer3.parameters():
            param.requires_grad = True
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True

    elif mode == "fully_trainable":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError("mode must be 'fully_frozen', 'partially_frozen', or 'fully_trainable'")

    return model

In [ ]:
# =========================
# Utility: show trainable vs frozen layers
# =========================

def print_layer_status(model):
    print("\nLayer-wise trainability status:")
    for name, child in model.named_children():
        total_params = sum(p.numel() for p in child.parameters())
        trainable_params = sum(p.numel() for p in child.parameters() if p.requires_grad)
        status = "Trainable" if trainable_params > 0 else "Frozen"
        print(f"{name:10s} | {status:10s} | Trainable params: {trainable_params:,} / {total_params:,}")

example_model = build_resnet50_transfer(NUM_CLASSES, mode="partially_frozen")
print_layer_status(example_model)

## 6. Training and Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, mode="partially_frozen"):
    model.train()
    
    # Keep frozen BatchNorm layers in eval mode for transfer learning stability
    if mode == "fully_frozen":
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
    elif mode == "partially_frozen":
        if hasattr(model, 'conv1'): model.conv1.eval()
        if hasattr(model, 'bn1'): model.bn1.eval()
        if hasattr(model, 'layer1'):
            for m in model.layer1.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        if hasattr(model, 'layer2'):
            for m in model.layer2.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()

    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)
        running_loss += loss.item() * images.size(0)
        running_corrects += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples
    return epoch_loss, epoch_acc

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    all_labels = []
    all_preds = []

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * images.size(0)
        running_corrects += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples
    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, num_epochs=10, patience=4, save_path="best_model.pth", mode="partially_frozen"):
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_loss = float("inf")
    best_model_wts = copy.deepcopy(model.state_dict())
    early_stop_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, mode=mode)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, save_path)
            early_stop_counter = 0
            print("Best model updated and saved.")
        else:
            early_stop_counter += 1
            print(f"No improvement. Early stop counter: {early_stop_counter}/{patience}")

        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_wts)
    return model, history

## 7. Hyperparameter Tuning

A small learning-rate tuning step is included before final training. Tuning runs for `TUNE_EPOCHS` epochs (or 1 epoch if in fast-run mode).

In [ ]:
def tune_learning_rate(mode, lr_list, train_loader, val_loader, num_classes, device):
    best_lr = None
    best_val_acc = -1.0

    print(f"\nTuning learning rate for mode: {mode}")

    epochs = 1 if FAST_RUN else TUNE_EPOCHS

    for lr in lr_list:
        print(f"\nTrying learning rate: {lr}")
        model = build_resnet50_transfer(num_classes, mode=mode).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr,
            weight_decay=WEIGHT_DECAY
        )
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=1
        )

        temp_save_path = os.path.join(MODEL_SAVE_DIR, f"temp_{mode}_{lr}.pth")

        model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=epochs,
            patience=2,
            save_path=temp_save_path,
            mode=mode
        )

        # Evaluate the loaded best weights on validation dataset
        _, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        print(f"Validation accuracy with lr={lr}: {val_acc:.4f}")

        if os.path.exists(temp_save_path):
            os.remove(temp_save_path)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_lr = lr

    print(f"\nBest LR for {mode}: {best_lr} | Best Val Acc: {best_val_acc:.4f}")
    return best_lr

## 8. Run Experiments

This section trains and compares:

- Fully frozen model
- Partially frozen model
- Fully trainable model

In [ ]:
def run_experiment(mode, train_loader, val_loader, test_loader, num_classes, class_names, device):
    best_lr = tune_learning_rate(
        mode=mode,
        lr_list=LR_CANDIDATES[mode],
        train_loader=train_loader,
        val_loader=val_loader,
        num_classes=num_classes,
        device=device
    )

    print(f"\nStarting final training for mode: {mode}")

    model = build_resnet50_transfer(num_classes, mode=mode).to(device)
    print_layer_status(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=best_lr,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )

    save_path = os.path.join(MODEL_SAVE_DIR, f"{DATASET_NAME}_{mode}_best.pth")
    epochs = 1 if FAST_RUN else MAX_EPOCHS
    patience = 1 if FAST_RUN else EARLY_STOPPING_PATIENCE

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=epochs,
        patience=patience,
        save_path=save_path,
        mode=mode
    )

    best_state_dict = torch.load(save_path, map_location=device)
    model.load_state_dict(best_state_dict)

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    print(f"\nFinal Test Loss ({mode}): {test_loss:.4f}")
    print(f"Final Test Acc  ({mode}): {test_acc:.4f}")

    return {
        "mode": mode,
        "model": model,
        "history": history,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "y_true": y_true,
        "y_pred": y_pred,
        "save_path": save_path,
        "best_lr": best_lr
    }

In [ ]:
results = {}

for mode in ["fully_frozen", "partially_frozen", "fully_trainable"]:
    results[mode] = run_experiment(
        mode=mode,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_classes=NUM_CLASSES,
        class_names=class_names,
        device=DEVICE
    )

## 9. Plot Training and Validation Curves

In [ ]:
def plot_history(history, title_prefix="Model"):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], marker='o', label="Train Loss")
    plt.plot(epochs, history["val_loss"], marker='o', label="Validation Loss")
    plt.title(f"{title_prefix} - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], marker='o', label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], marker='o', label="Validation Accuracy")
    plt.title(f"{title_prefix} - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
for mode in results:
    plot_history(results[mode]["history"], title_prefix=mode)

## 10. Confusion Matrix and Classification Report

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(12, 10))
    if len(class_names) <= 20:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
    else:
        sns.heatmap(cm, annot=False, cmap='Blues')
        plt.xticks([])
        plt.yticks([])

    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

In [ ]:
for mode in results:
    print(f"\n{'='*80}")
    print(f"Classification Report for {mode}")
    print(f"{'='*80}\n")

    y_true = results[mode]["y_true"]
    y_pred = results[mode]["y_pred"]

    report = classification_report(
        y_true, y_pred, target_names=class_names, digits=4, zero_division=0
    )
    print(report)

    plot_confusion_matrix(
        y_true, y_pred, class_names,
        title=f"{mode} - Confusion Matrix"
    )

## 11. Comparison Table

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [m for m in results.keys()],
    "Best Learning Rate": [results[m]["best_lr"] for m in results.keys()],
    "Test Loss": [results[m]["test_loss"] for m in results.keys()],
    "Test Accuracy": [results[m]["test_acc"] for m in results.keys()]
})

comparison_df = comparison_df.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison_df, x="Model", y="Test Accuracy", palette="viridis")
plt.title("Test Accuracy Comparison")
plt.ylim(0, 1)
plt.grid(axis="y")
plt.show()

## 12. Save and Load Model

In [ ]:
best_mode = comparison_df.iloc[0]["Model"]
loaded_model = build_resnet50_transfer(NUM_CLASSES, mode=best_mode).to(DEVICE)
loaded_model.load_state_dict(torch.load(
    os.path.join(MODEL_SAVE_DIR, f"{DATASET_NAME}_{best_mode}_best.pth"),
    map_location=DEVICE
))
loaded_model.eval()

print(f"Saved model loaded successfully. (Best Mode: {best_mode})")

## 13. Inference on Sample Test Images

In [ ]:
def show_predictions(model, loader, class_names, device, num_images=6):
    model.eval()
    images_shown = 0

    plt.figure(figsize=(12, 6))

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            for i in range(images.size(0)):
                if images_shown >= num_images:
                    plt.tight_layout()
                    plt.show()
                    return

                plt.subplot(2, 3, images_shown + 1)

                img = images[i].cpu().numpy().transpose((1, 2, 0))
                img = (img * np.array(IMAGENET_STD)) + np.array(IMAGENET_MEAN)
                img = np.clip(img, 0, 1)

                plt.imshow(img)
                plt.title(f"True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]}")
                plt.axis("off")

                images_shown += 1

    plt.tight_layout()
    plt.show()

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = results[best_model_name]["model"]

print(f"Showing predictions from best model: {best_model_name}")
show_predictions(best_model, test_loader, class_names, DEVICE, num_images=6)

## 14. Explanation and Comparison Analysis

### Why is ResNet-50 used?
ResNet-50 is a deep convolutional neural network with residual connections. It performs strongly on image classification and has high-quality pretrained ImageNet weights.

### Why freeze layers?
Early layers learn general features like edges and textures. Freezing them helps retain useful knowledge while reducing training cost.

### Effect of different freezing strategies
- **Fully Frozen**: Fastest, but may underfit
- **Partially Frozen**: Good balance between speed and performance
- **Fully Trainable**: Most flexible, but computationally expensive and may overfit

## 15. Conclusion

This notebook provided a complete transfer learning pipeline using **ResNet-50 pretrained on ImageNet** for **CIFAR-10 / CIFAR-100** classification in **PyTorch**. It included data preprocessing, model customization, freezing strategies, training, evaluation, plotting, model comparison, confusion matrix, classification report, early stopping, saving/loading, and inference.